In [1]:
pip install alpaca-py


   -------------------- ------------------- 1/2 [alpaca-py]
   ---------------------------------------- 2/2 [alpaca-py]

Note: you may need to restart the kernel to use updated packages.


In [1]:
from alpaca.trading.client import TradingClient
from dotenv import load_dotenv
import os
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from datetime import timedelta, datetime

load_dotenv()
API_KEY = os.getenv("API_KEY")
SECRET_KEY = os.getenv("SECRET_KEY")

client = TradingClient(API_KEY, SECRET_KEY, paper=True)
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

def get_live_bars(symbol, lookback=5):
    request = StockBarsRequest(
        symbol_or_symbols=symbol,
        timeframe=TimeFrame(5, TimeFrameUnit.Minute),
        start=datetime.now() - timedelta(days=lookback)
    )
    bars = data_client.get_stock_bars(request)
    return bars.df

account = client.get_account()
print(account)
df = get_live_bars("AAPL")
df  # This actually prints the dataframe in the correct format.  Better than both df.head() which prints the first 5 rows and print(df) oddly enough

id=UUID('c74df5a0-1b68-4c2c-a6af-dcf26673433d') account_number='PA3X8KRAB0TK' status=<AccountStatus.ACTIVE: 'ACTIVE'> crypto_status=<AccountStatus.ACTIVE: 'ACTIVE'> currency='USD' buying_power='398392.62' regt_buying_power='199196.31' daytrading_buying_power=None non_marginable_buying_power='99598.15' cash='99220.83' accrued_fees='0' pending_transfer_out=None pending_transfer_in=None portfolio_value='99975.48' pattern_day_trader=None trading_blocked=False transfers_blocked=False account_blocked=False created_at=datetime.datetime(2026, 6, 10, 16, 10, 57, 578916, tzinfo=TzInfo(0)) trade_suspended_by_user=False multiplier='4' shorting_enabled=True equity='99975.48' last_equity='99978.69' long_market_value='754.65' short_market_value='0' initial_margin='377.33' maintenance_margin='377.33' last_maintenance_margin='378.93' sma='99959.97' daytrade_count=None options_buying_power='99598.15' options_approved_level=3 options_trading_level=3


open      high       low     close  \
symbol timestamp                                                           
AAPL   2026-07-17 11:20:00+00:00  332.9500  333.0000  332.4900  332.4910   
       2026-07-17 11:25:00+00:00  332.5644  332.7700  332.5000  332.7500   
       2026-07-17 11:30:00+00:00  332.5200  332.7500  332.4700  332.5000   
       2026-07-17 11:35:00+00:00  332.6800  332.8000  332.5200  332.6900   
       2026-07-17 11:40:00+00:00  332.5201  333.0000  332.5201  332.7400   
...                                    ...       ...       ...       ...   
       2026-07-22 14:40:00+00:00  325.0000  325.0000  324.5000  324.8299   
       2026-07-22 14:45:00+00:00  324.7800  325.1025  324.7000  325.0750   
       2026-07-22 14:50:00+00:00  325.0800  325.4900  324.8200  325.2933   
       2026-07-22 14:55:00+00:00  325.3100  325.6200  325.0200  325.6000   
       2026-07-22 15:00:00+00:00  325.6000  325.6699  324.7200  324.8000   

                                    volume  trade_count        vwap  
symbol timestamp                                                     
AAPL   2026-07-17 11:20:00+00:00    6676.0        217.0  332.768555  
       2026-07-17 11:25:00+00:00    3686.0        140.0  332.690489  
       2026-07-17 11:30:00+00:00    4859.0        123.0  332.537518  
       2026-07-17 11:35:00+00:00    6633.0        334.0  332.725583  
       2026-07-17 11:40:00+00:00    5031.0        219.0  332.836082  
...                                    ...          ...         ...  
       2026-07-22 14:40:00+00:00  253002.0      11032.0  324.809010  
       2026-07-22 14:45:00+00:00  359557.0      11552.0  324.966931  
       2026-07-22 14:50:00+00:00  440811.0      13834.0  325.090251  
       2026-07-22 14:55:00+00:00  291156.0      11960.0  325.273552  
       2026-07-22 15:00:00+00:00  291504.0      12231.0  325.222245  

[619 rows x 7 columns]

In [4]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

def calculate_macd(df, fast=12, slow=26, signal=9):
    # calculate fast and slow EMAs
    df['ema_fast'] = df['close'].ewm(span=fast, adjust=False).mean()
    df['ema_slow'] = df['close'].ewm(span=slow, adjust=False).mean()
    
    # calculate MACD line (fast minus slow)
    df['macd'] = df['ema_fast'] - df['ema_slow']
    
    # calculate signal line (EMA of MACD line)
    df['signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    
    # calculate histogram (MACD minus signal)
    df['hist'] = df['macd'] - df['signal']

    return df

def plot_macd(df):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    # plot price on top
    ax1.plot(df.index, df['close'], label='Close')
    ax1.set_title('Price')
    ax1.legend()

    # plot MACD line and signal line in middle
    ax2.plot(df.index, df['macd'], label='MACD')
    ax2.plot(df.index, df['signal'], label='Signal')

    # plot histogram on bottom
    ax2.bar(df.index, df['hist'], label='Histogram', alpha=0.3)
    ax2.set_title('MACD')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    pass

# download data
# ticker = "AAPL"
# df = yf.download(ticker, start="2023-01-01", end="2024-01-01")

# calculate and plot
# df = calculate_macd(get_live_bars("AAPL"))
# plot_macd(df)

In [15]:
from datetime import datetime, time as dtime
from zoneinfo import ZoneInfo
from alpaca.common.exceptions import APIError
from alpaca.trading.requests import MarketOrderRequest
from alpaca.trading.enums import OrderSide, TimeInForce

def run_signals(df, symbol, qty=1, actionable=False):
    latest_hist = df['hist'].iloc[-1]
    previous_hist = df['hist'].iloc[-2]

    if previous_hist > 0 and latest_hist <= 0:
        action = 'SELL'
    elif previous_hist < 0 and latest_hist >= 0:
        action = 'BUY'
    else:
        action = 'HOLD'
    
    try:
        position = client.get_open_position(symbol)
        print(f"Currently holding {position.qty} shares of {symbol}")
        qty_held = int(position.qty)
    except APIError:
        print(f"No open position in {symbol}")
        qty_held = 0
    
    if action == 'SELL' and qty_held > 0:
        final_action = 'SELL'
    elif action == 'BUY' and qty_held == 0:
        final_action = 'BUY'
    else:
        final_action = 'HOLD'

    if final_action in ['BUY', 'SELL']:
        if actionable:
            side = OrderSide.BUY if final_action == 'BUY' else OrderSide.SELL
            order_data = MarketOrderRequest(
                symbol=symbol,
                qty=qty,
                side=side,
                time_in_force=TimeInForce.DAY
            )
            client.submit_order(order_data=order_data)
            print(f"Submitted {final_action} order for {qty} share(s) of {symbol}")
        else:
            print(f"[DRY RUN] Would {final_action} {qty} share(s) of {symbol}")

    return final_action


def is_market_open():
    now = datetime.now(ZoneInfo("America/New_York"))
    open_time = dtime(9, 30)
    close_time = dtime(16, 0)
    if now.weekday() < 5 and (open_time <= now.time() <= close_time):
        # check to see if there are market actions to take since the market is open
        return True
    return False

symbols = ["AAPL", "MSFT", "GOOGL", "JPM", "XOM"]

def job():
    if not is_market_open():
        return
    for symbol in symbols:
        
        df = get_live_bars(symbol)
        df = calculate_macd(df)
        print(run_signals(df, symbol, actionable=True))

In [16]:
job()

No open position in AAPL
HOLD
No open position in MSFT
HOLD
No open position in GOOGL
HOLD
No open position in JPM
HOLD
No open position in XOM
HOLD


In [11]:
df = calculate_macd(get_live_bars("AAPL"))
df.loc[df.index[-1], 'hist'] = -1
df.loc[df.index[-2], 'hist'] = 1
print(run_signals(df, "AAPL"))

No open position in AAPL
[DRY RUN] Would SELL 1 share(s) of AAPL
SELL
